# Step 3 — Model Boolean

**Tujuan praktikum**
1. Mahasiswa memahami model Boolean dan operator AND, OR, NOT.
2. Mahasiswa mampu mengimplementasikan algoritma *merge* (intersection) dua posting list terurut.
3. Mahasiswa mampu mengoptimasi urutan pemrosesan query berdasarkan *document frequency*.
4. Mahasiswa memahami keterbatasan model Boolean (tidak ada peringkat).

In [1]:
# Jalankan sekali jika pustaka belum terpasang:
# !pip install pandas numpy matplotlib Sastrawi
import pandas as pd
pd.set_option("display.max_colwidth", 120)

In [2]:
from stki_utils import corpus_texts, preprocess, build_inverted_index

docs = {d: preprocess(t) for d, t in corpus_texts().items()}
index = build_inverted_index(docs)
SEMUA_DOC = sorted(docs)

def postings(term):
    t = preprocess(term, use_stopword=False)
    return sorted(index.get(t[0], {})) if t else []

print("tani   :", postings("tani"))
print("air    :", postings("air"))

tani   : ['D07', 'D08', 'D09', 'D10', 'D11', 'D12', 'D23']
air    : ['D01', 'D07', 'D09', 'D10']


## Langkah 1 — Algoritma intersection (AND) dengan dua pointer
Karena posting list terurut, irisan dapat dihitung dalam waktu O(x + y) dengan menggeser dua pointer.

In [3]:
def intersect(p1, p2):
    hasil, i, j, perbandingan = [], 0, 0, 0
    while i < len(p1) and j < len(p2):
        perbandingan += 1
        if p1[i] == p2[j]:
            hasil.append(p1[i]); i += 1; j += 1
        elif p1[i] < p2[j]:
            i += 1
        else:
            j += 1
    return hasil, perbandingan

hasil, n = intersect(postings("petani"), postings("air"))
print("petani AND air =", hasil, f"({n} perbandingan)")

petani AND air = ['D07', 'D09', 'D10'] (5 perbandingan)


## Langkah 2 — Union (OR) dan NOT

In [4]:
def union(p1, p2):
    hasil, i, j = [], 0, 0
    while i < len(p1) and j < len(p2):
        if p1[i] == p2[j]:
            hasil.append(p1[i]); i += 1; j += 1
        elif p1[i] < p2[j]:
            hasil.append(p1[i]); i += 1
        else:
            hasil.append(p2[j]); j += 1
    return hasil + p1[i:] + p2[j:]

def and_not(p1, p2):
    s2 = set(p2)
    return [d for d in p1 if d not in s2]

print("pupuk OR kompos     =", union(postings("pupuk"), postings("kompos")))
print("tani AND NOT hama   =", and_not(postings("tani"), postings("hama")))

pupuk OR kompos     = ['D08', 'D10']
tani AND NOT hama   = ['D08', 'D09', 'D10', 'D12', 'D23']


## Langkah 3 — Optimasi query: proses term dengan df terkecil lebih dulu
Untuk query `t1 AND t2 AND ... AND tn`, hasil antara tidak mungkin lebih besar dari posting list terpendek. Mengurutkan term berdasarkan df menaikkan efisiensi.

In [5]:
def and_query(terms, optimasi=True):
    lists = [postings(t) for t in terms]
    if optimasi:
        lists.sort(key=len)
    hasil, total = lists[0], 0
    for p in lists[1:]:
        hasil, n = intersect(hasil, p)
        total += n
    return hasil, total

query = ["data", "aplikasi", "sensor"]
print("df:", {t: len(postings(t)) for t in query})
print("Tanpa optimasi :", and_query(query, optimasi=False))
print("Dengan optimasi:", and_query(query, optimasi=True))

df: {'data': 6, 'aplikasi': 3, 'sensor': 2}
Tanpa optimasi : (['D10'], 8)
Dengan optimasi: (['D10'], 5)


## Langkah 4 — Parser query Boolean lengkap
`stki_utils.boolean_search` mem-parsing query dengan operator `AND`, `OR`, `NOT` (huruf kapital) dan tanda kurung, dengan prioritas NOT > AND > OR. Pelajari kodenya di `stki_utils.py`.

In [5]:
from stki_utils import boolean_search

contoh_query = [
    "petani AND pupuk",
    "pupuk OR kompos",
    "tani AND NOT hama",
    "(digital OR daring) AND internet",
    "gizi AND (anak OR balita) AND NOT sekolah",
]
for q in contoh_query:
    print(f"{q:45s} -> {boolean_search(q, index, SEMUA_DOC)}")

petani AND pupuk                              -> ['D08', 'D10']
pupuk OR kompos                               -> ['D08', 'D10']
tani AND NOT hama                             -> ['D08', 'D09', 'D10', 'D12', 'D23']
(digital OR daring) AND internet              -> ['D25']
gizi AND (anak OR balita) AND NOT sekolah     -> ['D02']


## Langkah 5 — Keterbatasan model Boolean
Perhatikan dua query berikut. Query AND terlalu ketat (sering kosong), sedangkan query OR terlalu longgar dan semua hasil dianggap sama relevannya karena tidak ada peringkat.

In [7]:
q = "keamanan data pribadi di internet"
terms = preprocess(q)
print("Term query:", terms)
print("AND :", boolean_search(" AND ".join(terms), index, SEMUA_DOC))
print("OR  :", boolean_search(" OR ".join(terms), index, SEMUA_DOC))

Term query: ['aman', 'data', 'pribadi', 'internet']
AND : []
OR  : ['D06', 'D10', 'D14', 'D15', 'D16', 'D18', 'D25', 'D27']


## Tugas Step 3
1. Implementasikan fungsi `or_not(p1, p2)` untuk `t1 OR NOT t2` menggunakan daftar semua dokumen. Mengapa operasi ini mahal pada koleksi besar?
2. Buat 5 query Boolean untuk mencari dokumen topik **ekonomi** tanpa mengambil dokumen topik lain. Laporkan hasil tiap query dan jelaskan strategi Anda.
3. Tambahkan dukungan *phrase query* (dalam tanda kutip) ke parser Boolean dengan memanfaatkan positional index dari Step 2.
4. Diskusikan: mengapa model Boolean masih digunakan pada pencarian dokumen hukum atau paten?

In [8]:
# Tulis jawaban Anda di sini